In [ ]:
import pandas as pd
import numpy as np
df = pd.DataFrame({
    "年份": [2023,2023,2023,2023,2024,2024,2024,2024],
    "季度": ["Q1","Q1","Q2","Q2","Q1","Q1","Q2","Q2"],
    "门店": ["A","B","A","B","A","B","A","B"],
    "销售额": [100,120,110,130,150,140,160,170],
})
mi = df.set_index(['年份','季度','门店']).sort_index()
print(mi)
print(mi.loc[(2023,'Q1')])
print(mi.loc[(2023,slice(None),'A'),:])
print(mi.xs("A",level="门店"))
print(mi.xs(2023,level="年份",drop_level=False))

            销售额
年份   季度 门店     
2023 Q1 A   100
        B   120
     Q2 A   110
        B   130
2024 Q1 A   150
        B   140
     Q2 A   160
        B   170
    销售额
门店     
A   100
B   120
            销售额
年份   季度 门店     
2023 Q1 A   100
     Q2 A   110
         销售额
年份   季度     
2023 Q1  100
     Q2  110
2024 Q1  150
     Q2  160
            销售额
年份   季度 门店     
2023 Q1 A   100
        B   120
     Q2 A   110
        B   130


In [10]:
print(mi.swaplevel('季度','门店').sort_index())
print(mi.reorder_levels(['门店','年份','季度']).sort_index())
print(mi.reset_index(level='季度'))
print(mi.index.names,mi.index.levels)

            销售额
年份   门店 季度     
2023 A  Q1  100
        Q2  110
     B  Q1  120
        Q2  130
2024 A  Q1  150
        Q2  160
     B  Q1  140
        Q2  170
            销售额
门店 年份   季度     
A  2023 Q1  100
        Q2  110
   2024 Q1  150
        Q2  160
B  2023 Q1  120
        Q2  130
   2024 Q1  140
        Q2  170
         季度  销售额
年份   门店         
2023 A   Q1  100
     B   Q1  120
     A   Q2  110
     B   Q2  130
2024 A   Q1  150
     B   Q1  140
     A   Q2  160
     B   Q2  170
['年份', '季度', '门店'] [[2023, 2024], ['Q1', 'Q2'], ['A', 'B']]


In [22]:
print(mi.groupby(level=['年份','门店']).sum())
print(mi.groupby(level='年份').sum())
wide = mi['销售额'].unstack('门店')
print(wide)
print(wide.stack())
print(wide.stack().unstack('年份'))
pv = df.pivot_table(index='年份',columns='季度',values='销售额',aggfunc=['sum','mean'])
print(pv)
print(pv['sum'])
print(pv.loc[2023,('mean','Q1')])

         销售额
年份   门店     
2023 A   210
     B   250
2024 A   310
     B   310
      销售额
年份       
2023  460
2024  620
门店         A    B
年份   季度          
2023 Q1  100  120
     Q2  110  130
2024 Q1  150  140
     Q2  160  170
年份    季度  门店
2023  Q1  A     100
          B     120
      Q2  A     110
          B     130
2024  Q1  A     150
          B     140
      Q2  A     160
          B     170
dtype: int64
年份     2023  2024
季度 门店            
Q1 A    100   150
   B    120   140
Q2 A    110   160
   B    130   170
      sum        mean       
季度     Q1   Q2     Q1     Q2
年份                          
2023  220  240  110.0  120.0
2024  290  330  145.0  165.0
季度     Q1   Q2
年份            
2023  220  240
2024  290  330
110.0


In [26]:
orders = pd.DataFrame({
    "订单时间": pd.to_datetime(["2024-01-10 10:00", "2024-01-15 09:00", "2024-02-01 14:00"]),
    "用户ID": [1, 2, 1], "金额": [100, 200, 150],
})
price_log = pd.DataFrame({
    "生效时间": pd.to_datetime(["2024-01-01", "2024-01-12", "2024-01-20", "2024-02-05"]),
    "单价":     [10, 12, 11, 13],
})
out = pd.merge_asof(
    orders.sort_values('订单时间'),
    price_log.sort_values('生效时间'),
    left_on='订单时间',right_on='生效时间',
    direction='backward',
    tolerance=pd.Timedelta('30D'),
)
print(out)

                 订单时间  用户ID   金额       生效时间  单价
0 2024-01-10 10:00:00     1  100 2024-01-01  10
1 2024-01-15 09:00:00     2  200 2024-01-12  12
2 2024-02-01 14:00:00     1  150 2024-01-20  11
